# Overnight replication: two new seeds, stronger controls

Extract **all supplied files together**. This notebook launches a detached worker;
it does not continuously print progress. Refresh the status cell whenever wanted.

Priority: both seeds' main conditions → five matched random schedules per budget
→ event-local history controls → optional historical curvature audit → deferred frames.
No existing experiment is modified. Read `README.md` for exact definitions.

In [ ]:
from pathlib import Path
import sys, json, subprocess
import overnight_study as study

candidate = Path('/home/ubuntu/4/env_qwen3/bin/python')
WORKER_PYTHON = str(candidate) if candidate.exists() else sys.executable
probe = subprocess.run([WORKER_PYTHON, '-c',
    "import torch,transformers,datasets,json; print(json.dumps(dict(torch=torch.__version__,transformers=transformers.__version__,cuda=torch.cuda.is_available())))"], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError(probe.stderr[-4000:] + '\nUse the compatible environment from the previous run. requirements.txt lists the extra packages; no upgrades are automatic.')
print('Worker:', WORKER_PYTHON)
print(probe.stdout.strip().splitlines()[-1])


## Settings — fixed before launch

Defaults: Pythia-410M, seeds 12 and 13, SST-2 → AG News, 600 A steps,
128 B steps, unchanged frozen predictors, an 11.5-hour session.
The complete queue may need another session; saved units resume.

The old curvature re-audit needs the original **full run folder**, including its
A-trained checkpoint. If it is found below, the optional re-audit is enabled.
A share ZIP alone cannot reconstruct its weights. New-seed experiments work
without that optional source.

In [ ]:
SETTINGS = study.defaults(Path.cwd() / 'runs' / 'overnight_replication_v1')
SETTINGS['python'] = WORKER_PYTHON
SETTINGS['hours'] = 11.5

# The expensive fixed-rule frame/path replication comes LAST.
SETTINGS['deferred_frames'] = True

candidates = [
    Path('/home/ubuntu/5/runs/reviewer_overnight_seed11_v1'),
    Path.home() / '5/runs/reviewer_overnight_seed11_v1',
    Path.cwd() / 'runs/reviewer_overnight_seed11_v1',
]
for source in candidates:
    anchor = source / 'checkpoints/pythia410m/text/seed11/anchor.pt'
    data = source / 'data/pythia410m/text/seed11.json'
    state = source / 'status.json'
    if anchor.exists() and data.exists() and state.exists() and (source / 'results.sqlite').exists():
        if json.loads(state.read_text()).get('status') == 'complete':
            SETTINGS['curvature_reaudit_source'] = str(source.resolve())
            break

# Or set an alternative full source directory here BEFORE first launch:
# SETTINGS['curvature_reaudit_source'] = '/actual/path/to/reviewer_overnight_seed11_v1'

print('Seeds:', SETTINGS['seeds'], '| output:', SETTINGS['output'])
print('First stage: 12 main trajectories; later stages resume after the time limit.')
print('Historical curvature source:', SETTINGS['curvature_reaudit_source'] or 'not found; optional historical re-audit disabled')


## Launch / resume

Run once. Repeating this while the worker is active does not launch another worker.
After a budget pause, this same cell resumes. No scrolling progress output.

In [ ]:
_ = study.launch(SETTINGS)


## Refresh status manually

`stage` tells you which priority is running. `alive` checks the worker lock.
`seconds_since_progress` is not reset by refreshing this cell. An active worker
can be busy inside a long operation; liveness does not prove recent progress.

In [ ]:
print(json.dumps(study.status(SETTINGS), indent=2))


## Numbers and plots

The report keeps old-task retention and new-task learning side by side. Random
schedule variation is reported within each seed, without treating schedules as
independent training seeds. Incomplete units stay labelled.

In [ ]:
# Uncomment when you want the report and plots:
# study.show_results(SETTINGS)


## Stop / restart

Stop is cooperative. Wait for `alive=False` before restarting. Resume reuses
completed work; restart creates a new directory and preserves the old run.

In [ ]:
# study.stop(SETTINGS)
# To resume, run the Launch/resume cell above.
# For a fresh run after the worker stops:
# SETTINGS = study.restart(SETTINGS)


## Compact share ZIP

This includes a consistent database snapshot, reports and code. It omits model
and optimizer checkpoints, which remain in the full run directory.

In [ ]:
# ZIP_PATH = study.export(SETTINGS)
# print(ZIP_PATH)


## Optional CPU smoke check

Separate tiny randomly initialized model; software validation only.
It does not supply paper results and does not require downloaded model weights.

In [ ]:
# SMOKE = study.smoke_settings(Path.cwd() / 'runs' / 'overnight_smoke')
# SMOKE['python'] = WORKER_PYTHON
# _ = study.launch(SMOKE)
# print(study.status(SMOKE))
